## Hyperparameter Tuning with Torch and Ray Tune

Author: Nik Alleyne Author Blog: https://www.securitynik.com Author GitHub: github.com/securitynik

Author Books: [

        "https://www.amazon.ca/Learning-Practicing-Leveraging-Practical-Detection/dp/1731254458/", <br>
        "https://www.amazon.ca/Learning-Practicing-Mastering-Network-Forensics/dp/1775383024/" <br>,
        "https://drive.google.com/file/d/1Ry5r659n8G9Hk8Z8pSKLtqrYyWl8ipbs/view"
    ] 

In [22]:
# let's get our libraries
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

In [23]:
# Get the datasets
X, y = make_classification(
    n_samples=5000,
    n_features=6,
    n_informative=4,
    n_redundant=2,
    n_classes=3,
    n_clusters_per_class=2,
    weights=None,
    class_sep=1.0,
    random_state=42
)

print(f'Raw data shape: X: {X.shape} -> y: {y.shape}')

# With the data in place, let's split it now
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, train_size=0.7, stratify=y, random_state=42)

print(f'Train shape: {X_train.shape} -> y: {y_train.shape}')

# tmp is not splitted into test and validation
X_test, X_val, y_test, y_val = train_test_split(X_tmp, y_tmp, train_size=0.5, stratify=y_tmp, random_state=10)

# Now take the test 
print(f'After splitting: X_test: {X_test.shape} -> y test: {y_test.shape} ')
print(f'Validation Split: X_val: {X_val.shape} -> y test: {y_val.shape} ')

Raw data shape: X: (5000, 6) -> y: (5000,)
Train shape: (3500, 6) -> y: (3500,)
After splitting: X_test: (750, 6) -> y test: (750,) 
Validation Split: X_val: (750, 6) -> y test: (750,) 


In [24]:
# With the data in place, let's scale the data via standardization
from sklearn.preprocessing import StandardScaler

In [25]:
# Setup the scaler
scaler = StandardScaler()

# Scale the data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_val_scaled = scaler.transform(X_val)

# What is the mean and standard deviation of the training data
print(f'Training data mean: {X_train_scaled.mean()}')
print(f'Training data mean: {X_train_scaled.std()}')

Training data mean: -3.4173722053223866e-17
Training data mean: 1.0000000000000002


In [26]:
# Our ys are currently 0, 1, 2 for the 3 classes
# As a result, we will use Categorical Crossentropy class
np.unique(y)

array([0, 1, 2])

In [27]:
# Let us also prepare the data for PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F

In [28]:
# Let's ensure Torch can also use the GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'We are running on: {device.upper()}')

We are running on: CUDA


In [29]:
# Setup the torch data now
X_train_pt = torch.as_tensor(X_train_scaled, dtype=torch.float32)
X_test_pt = torch.as_tensor(X_test_scaled, dtype=torch.float32)
X_val_pt = torch.as_tensor(X_val_scaled, dtype=torch.float32)

# Setup the torch targets
y_train_pt = torch.as_tensor(y_train, dtype=torch.long, device=device)
y_test_pt = torch.as_tensor(y_test, dtype=torch.long, device=device)
y_val_pt = torch.as_tensor(y_val, dtype=torch.long, device=device)

In [30]:
# Define the output shape
NUM_CLASSES = 3

# input dimension
INPUT_DIM = X.shape[1]
NUM_CLASSES, INPUT_DIM

(3, 6)

In [31]:
# Define the search space
# These are the values for which Hyperband should find the best combination
SEARCH_SPACE = {
    "num_layers": [1, 2, 3],
    "units_per_layer": [32, 64, 128],
    "activation": ["relu", "tanh", "elu"],
    "dropout_rate": [0.0, 0.1, 0.2, 0.4],
    "learning_rate": [1e-4, 3e-4, 1e-3],
}

# with this search space, how many possible combinations we have
num_possible_configs = np.prod([ len(value) for value in SEARCH_SPACE.values() ])

print(f'Possible configurations: {num_possible_configs} ')

Possible configurations: 324 


In [32]:
# Setup a sample config
example_config = {
    "num_layers": 2,
    "units_per_layer": 64,
    "activation": "relu",
    "dropout_rate": 0.2,
    "learning_rate": 1e-3,
}

In [33]:
# Define the training config
# These are fixed
TRAINING_CONFIG = {
    "max_epochs": 3,
    "batch_size": 64,
    "random_seed": 42,
}


In [34]:
# Let's create a PyTorch class
class TorchClassifierTuner(nn.Module):
    def __init__(
        self, 
        input_dim, 
        num_classes, 
        num_layers,
        units_per_layer, 
        activation, 
        dropout_rate
    ):
        super().__init__()

        activation_functions = {
            'relu' : nn.ReLU(),
            'tanh' : nn.Tanh(),
            'elu' : nn.ELU()
        }

        layers = []
        current_dim = input_dim
        activation_class = activation_functions[activation]

        for layer_idx in range(num_layers):
            layers.append(
                nn.Linear(
                    current_dim, 
                    units_per_layer
                )
            )
            layers.append(activation_class)

            if dropout_rate > 0.0:
                layers.append(
                    nn.Dropout(p=dropout_rate)
                )

            current_dim = units_per_layer

        layers.append(
            nn.Linear(
                current_dim,
                num_classes
            )
        )

        # Construct the network
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


In [35]:
# Let get the libraries so we can setup our dataloader
from torch.utils.data import DataLoader, TensorDataset

In [36]:
# Build the datasets
train_dataset_pt = TensorDataset(
    X_train_pt,
    y_train_pt
)

X_val_dataset_pt = TensorDataset(
    X_val_pt,
    y_val_pt
)

test_dataset_pt = TensorDataset(
    X_test_pt,
    y_test_pt
)

# Look at a sample fo the train dataset
next(iter(train_dataset_pt))

(tensor([ 0.4636, -0.5744,  0.7802,  1.0506,  1.4558, -0.6825]),
 tensor(0, device='cuda:0'))

In [37]:
# Let's setup our ray tune
from ray import tune

# As we are here, let's also add model checkpointing
# This allows us to save the model, whenever a trial reaches a reporting point
import tempfile
from ray.tune import Checkpoint
import os

In [38]:
# Let us setup a function to train our torch model
def train_model_pt(config):
    seed = TRAINING_CONFIG['random_seed']

    np.random.seed(seed=seed)
    torch.manual_seed(seed=seed)

    # Build the model
    model = TorchClassifierTuner(
        input_dim=INPUT_DIM,
        num_classes=NUM_CLASSES,
        num_layers=config['num_layers'],
        units_per_layer=config['units_per_layer'],
        activation=config['activation'],
        dropout_rate=config['dropout_rate']
        ).to(device)

    # Define the loss function
    loss_fn = nn.CrossEntropyLoss()

    # Define the optimizer
    optimizer = torch.optim.Adam(
        params=model.parameters(),
        lr=config['learning_rate']
    )

    # Create a train loader
    train_loader = DataLoader(
        train_dataset_pt,
        batch_size=TRAINING_CONFIG['batch_size'],
        shuffle=True
    )

    # Validation set
    validation_loader = DataLoader(
        X_val_dataset_pt,
        batch_size=TRAINING_CONFIG['batch_size'],
        shuffle=False
    )

    for epoch in range(TRAINING_CONFIG['max_epochs']):
        # Train the model
        model.train()

        total_train_loss = 0.0
        train_examples = 0.

        # Get the data and move them to the device
        for X, y in train_loader:
            X = X.to(device)
            y = y.to(device)

            # Zero out the optimizer
            optimizer.zero_grad(set_to_none=True)

            # Get the logits
            logits = model(X)

            # Calculate the loss
            loss = loss_fn(logits, y)

            # Perform backpropagation
            loss.backward()

            # Perform gradient descent
            optimizer.step()

            # Get the batch_size
            batch_size = y.size(0)

            # Calculate the total train loss
            total_train_loss += loss.item() * batch_size
            train_examples += batch_size

        train_loss = total_train_loss / train_examples

        # Evaluate the model's performance
        model.eval()
        total_val_loss = 0.0
        validation_examples = 0
        correct = 0

        with torch.no_grad():
            for X, y in validation_loader:
                X = X.to(device)
                y = y.to(device)

                logits = model(X)
                loss = loss_fn(logits, y)

                batch_size = y.size(0)
                total_val_loss += loss.item() * batch_size
                validation_examples += batch_size

                predictions = torch.argmax(logits, dim=-1)
                correct += (predictions == y).sum().item()

        # Get the validation loss
        validation_loss = (
            total_val_loss / validation_examples
        )

        # Get the validation accuracy
        validation_accuracy = (
            correct / validation_examples
        )

        # Let's also store the checkpoints
        with tempfile.TemporaryDirectory() as checkpoint_dir:
            checkpoint_path = os.path.join(
                checkpoint_dir,
                'model.pt'
            )

            torch.save(
                {
                    'epoch' : epoch,
                    'model_state_dict' : model.state_dict(),
                    'optimizer_state_dict' : optimizer.state_dict(),
                    'config' : config,
                    'val_accuracy' : validation_accuracy
                },
                f=checkpoint_path
            )


            # Provide the input to Ray, so it can decide on how to handle the trial
            tune.report(
                {
                'train_loss' : train_loss,
                'val_loss' : validation_loss,
                'val_accuracy' : validation_accuracy
                },
                checkpoint=Checkpoint.from_directory(
                    path=checkpoint_dir
                )
            )        
            
        print(f'Epoch: {epoch + 1} | train loss: {train_loss} | eval loss: {validation_loss} | val accuracy: {validation_accuracy}')

    # In a typical PyTorch training, we would return the model
    # not in this case
    # return model

In [39]:
# Connecting the search space to RayTune and preparing to run Hyperband
from ray.tune.schedulers import HyperBandScheduler
from ray.tune import RunConfig

In [40]:
# Convert out existing search space dictionary into RayTune choice
SEARCH_SPACE

{'num_layers': [1, 2, 3],
 'units_per_layer': [32, 64, 128],
 'activation': ['relu', 'tanh', 'elu'],
 'dropout_rate': [0.0, 0.1, 0.2, 0.4],
 'learning_rate': [0.0001, 0.0003, 0.001]}

In [41]:
# This is the new Ray search space
RAY_SEARCH_SPACE = {
    'num_layers' : tune.choice(
        SEARCH_SPACE['num_layers']
        ),
    'units_per_layer' : tune.choice(
        SEARCH_SPACE['units_per_layer']
    ),
    'activation' : tune.choice(
        SEARCH_SPACE['activation']
    ),
    'dropout_rate' : tune.choice(
        SEARCH_SPACE['dropout_rate']
    ),
    'learning_rate' : tune.choice(
        SEARCH_SPACE['learning_rate']
    )
}

# The new Ray search space
RAY_SEARCH_SPACE

{'num_layers': <ray.tune.search.sample.Categorical at 0x765448f46c00>,
 'units_per_layer': <ray.tune.search.sample.Categorical at 0x765448f46c60>,
 'activation': <ray.tune.search.sample.Categorical at 0x765771ca91f0>,
 'dropout_rate': <ray.tune.search.sample.Categorical at 0x765771ca90d0>,
 'learning_rate': <ray.tune.search.sample.Categorical at 0x765771ca8530>}

In [42]:
seed = TRAINING_CONFIG['random_seed']
np.random.seed(seed=seed)
torch.manual_seed(seed=seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

g = torch.Generator()
g.manual_seed(42)

# Now let's get the hyperband scheduler
hyperband_scheduler = HyperBandScheduler(
    time_attr='training_iteration',
    metric='val_accuracy',
    mode='max',
    max_t=TRAINING_CONFIG['max_epochs'],
    reduction_factor=3
)


# We are finally here, let's put together the tuner
ray_tuner = tune.Tuner(
    train_model_pt,
    param_space=RAY_SEARCH_SPACE,
    tune_config=tune.TuneConfig(
        scheduler=hyperband_scheduler,
        num_samples=20 # The amount of sample configurations to run
    ),
    run_config=RunConfig(
        name='torch_hyperband_classifier',
        storage_path='/tmp/ray_results',
    )
    
)

# Fit the tuner
ray_results = ray_tuner.fit()
ray_results

(train_model_pt pid=17241) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/tmp/ray_results/torch_hyperband_classifier/train_model_pt_3809c_00003_3_activation=tanh,dropout_rate=0.4000,learning_rate=0.0003,num_layers=2,units_per_layer=32_2026-09-17_23-51-24/checkpoint_000000)


(train_model_pt pid=17243) Epoch: 1 | train loss: 1.0791434583663941 | eval loss: 1.057222903251648 | val accuracy: 0.4266666666666667


(train_model_pt pid=17630) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/tmp/ray_results/torch_hyperband_classifier/train_model_pt_3809c_00010_10_activation=tanh,dropout_rate=0.1000,learning_rate=0.0001,num_layers=1,units_per_layer=64_2026-09-17_23-51-24/checkpoint_000000) [repeated 18x across cluster]


(train_model_pt pid=17630) Epoch: 1 | train loss: 1.0634530890328544 | eval loss: 1.0361374390920004 | val accuracy: 0.568 [repeated 10x across cluster]


(train_model_pt pid=18044) Restored on 10.0.0.45 from checkpoint: Checkpoint(filesystem=local, path=/tmp/ray_results/torch_hyperband_classifier/train_model_pt_3809c_00007_7_activation=elu,dropout_rate=0.1000,learning_rate=0.0010,num_layers=2,units_per_layer=64_2026-09-17_23-51-24/checkpoint_000000)
(train_model_pt pid=17652) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/tmp/ray_results/torch_hyperband_classifier/train_model_pt_3809c_00015_15_activation=tanh,dropout_rate=0.1000,learning_rate=0.0003,num_layers=2,units_per_layer=64_2026-09-17_23-51-25/checkpoint_000002) [repeated 13x across cluster]


(train_model_pt pid=18020) Epoch: 1 | train loss: 0.8740295658792768 | eval loss: 0.7599727374712626 | val accuracy: 0.66 [repeated 6x across cluster]


(train_model_pt pid=18269) Restored on 10.0.0.45 from checkpoint: Checkpoint(filesystem=local, path=/tmp/ray_results/torch_hyperband_classifier/train_model_pt_3809c_00014_14_activation=tanh,dropout_rate=0.0000,learning_rate=0.0010,num_layers=2,units_per_layer=64_2026-09-17_23-51-25/checkpoint_000000)
(train_model_pt pid=18045) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/tmp/ray_results/torch_hyperband_classifier/train_model_pt_3809c_00019_19_activation=relu,dropout_rate=0.0000,learning_rate=0.0010,num_layers=1,units_per_layer=32_2026-09-17_23-51-25/checkpoint_000002) [repeated 10x across cluster]


(train_model_pt pid=18269) Epoch: 1 | train loss: 0.8449617481231689 | eval loss: 0.7193905329704284 | val accuracy: 0.676 [repeated 5x across cluster]


2026-09-17 23:52:21,889	INFO tune.py:1007 -- Wrote the latest version of all result files and experiment state to '/tmp/ray_results/torch_hyperband_classifier' in 0.0115s.
2026-09-17 23:52:21,904	INFO tune.py:1039 -- Total run time: 57.31 seconds (57.25 seconds for the tuning loop).


ResultGrid<[
  Result(
    metrics={'train_loss': 0.9923138743809291, 'val_loss': 0.9654912767410279, 'val_accuracy': 0.5453333333333333},
    path='/tmp/ray_results/torch_hyperband_classifier/train_model_pt_3809c_00000_0_activation=elu,dropout_rate=0.2000,learning_rate=0.0001,num_layers=3,units_per_layer=32_2026-09-17_23-51-24',
    filesystem='local',
    checkpoint=Checkpoint(filesystem=local, path=/tmp/ray_results/torch_hyperband_classifier/train_model_pt_3809c_00000_0_activation=elu,dropout_rate=0.2000,learning_rate=0.0001,num_layers=3,units_per_layer=32_2026-09-17_23-51-24/checkpoint_000002)
  ),
  Result(
    metrics={'train_loss': 0.702030024937221, 'val_loss': 0.6816801056861878, 'val_accuracy': 0.7093333333333334},
    path='/tmp/ray_results/torch_hyperband_classifier/train_model_pt_3809c_00001_1_activation=tanh,dropout_rate=0.2000,learning_rate=0.0010,num_layers=1,units_per_layer=128_2026-09-17_23-51-24',
    filesystem='local',
    checkpoint=Checkpoint(filesystem=local, pa

In [43]:
# Get the best results
best_result = ray_results.get_best_result(
    metric='val_accuracy',
    mode='max'
)

# Get the best result config
for key, value in best_result.config.items():
    print(f'{key}:{value}')

num_layers:3
units_per_layer:128
activation:relu
dropout_rate:0.4
learning_rate:0.001


In [44]:
# Get the metrics
best_result.metrics

{'train_loss': 0.48176136268888203,
 'val_loss': 0.41493727795283,
 'val_accuracy': 0.8613333333333333,
 'timestamp': 1789703503,
 'checkpoint_dir_name': 'checkpoint_000002',
 'should_checkpoint': True,
 'done': True,
 'training_iteration': 3,
 'trial_id': '3809c_00002',
 'date': '2026-09-17_23-51-43',
 'time_this_iter_s': 0.5031967163085938,
 'time_total_s': 6.530235528945923,
 'pid': 17238,
 'hostname': 'SECURITYNIK-G14',
 'node_ip': '10.0.0.45',
 'config': {'num_layers': 3,
  'units_per_layer': 128,
  'activation': 'relu',
  'dropout_rate': 0.4,
  'learning_rate': 0.001},
 'time_since_restore': 6.530235528945923,
 'iterations_since_restore': 3,
 'experiment_tag': '2_activation=relu,dropout_rate=0.4000,learning_rate=0.0010,num_layers=3,units_per_layer=128'}

In [45]:
# List all completed trials
for result in ray_results:
    print(
        result.config,
        result.metrics.get('val_accuracy')
    )

{'num_layers': 3, 'units_per_layer': 32, 'activation': 'elu', 'dropout_rate': 0.2, 'learning_rate': 0.0001} 0.5453333333333333
{'num_layers': 1, 'units_per_layer': 128, 'activation': 'tanh', 'dropout_rate': 0.2, 'learning_rate': 0.001} 0.7093333333333334
{'num_layers': 3, 'units_per_layer': 128, 'activation': 'relu', 'dropout_rate': 0.4, 'learning_rate': 0.001} 0.8613333333333333
{'num_layers': 2, 'units_per_layer': 32, 'activation': 'tanh', 'dropout_rate': 0.4, 'learning_rate': 0.0003} 0.56
{'num_layers': 2, 'units_per_layer': 64, 'activation': 'relu', 'dropout_rate': 0.0, 'learning_rate': 0.0003} 0.5586666666666666
{'num_layers': 2, 'units_per_layer': 32, 'activation': 'relu', 'dropout_rate': 0.0, 'learning_rate': 0.001} 0.8066666666666666
{'num_layers': 3, 'units_per_layer': 128, 'activation': 'tanh', 'dropout_rate': 0.4, 'learning_rate': 0.001} 0.8
{'num_layers': 2, 'units_per_layer': 64, 'activation': 'elu', 'dropout_rate': 0.1, 'learning_rate': 0.001} 0.748
{'num_layers': 3, 'uni

In [46]:
best_result.checkpoint

Checkpoint(filesystem=local, path=/tmp/ray_results/torch_hyperband_classifier/train_model_pt_3809c_00002_2_activation=relu,dropout_rate=0.4000,learning_rate=0.0010,num_layers=3,units_per_layer=128_2026-09-17_23-51-24/checkpoint_000002)

In [47]:
# With all of this in place, let's put together our final PyTorch model based on the best config
best_config = best_result.config
best_config

{'num_layers': 3,
 'units_per_layer': 128,
 'activation': 'relu',
 'dropout_rate': 0.4,
 'learning_rate': 0.001}

In [48]:
seed = seed = TRAINING_CONFIG['random_seed']
np.random.seed(seed=seed)
torch.manual_seed(seed=seed)

model_pt = TorchClassifierTuner(
    input_dim=INPUT_DIM,
    num_classes=NUM_CLASSES,
    num_layers=best_config['num_layers'],
    units_per_layer=best_config['units_per_layer'],
    activation=best_config['activation'],
    dropout_rate=best_config['dropout_rate']
    )

model_pt



TorchClassifierTuner(
  (network): Sequential(
    (0): Linear(in_features=6, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.4, inplace=False)
    (3): Linear(in_features=128, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.4, inplace=False)
    (6): Linear(in_features=128, out_features=128, bias=True)
    (7): ReLU()
    (8): Dropout(p=0.4, inplace=False)
    (9): Linear(in_features=128, out_features=3, bias=True)
  )
)

In [50]:
# Get the model summary
import torchinfo

torchinfo.summary(model_pt)

Layer (type:depth-idx)                   Param #
TorchClassifierTuner                     --
├─Sequential: 1-1                        --
│    └─Linear: 2-1                       896
│    └─ReLU: 2-2                         --
│    └─Dropout: 2-3                      --
│    └─Linear: 2-4                       16,512
│    └─ReLU: 2-5                         --
│    └─Dropout: 2-6                      --
│    └─Linear: 2-7                       16,512
│    └─ReLU: 2-8                         --
│    └─Dropout: 2-9                      --
│    └─Linear: 2-10                      387
Total params: 34,307
Trainable params: 34,307
Non-trainable params: 0

In [51]:
# We could have also confirmed the number of trainable parameters by simply doing
num_parsms = sum([ p.numel() for p in model_pt.parameters() ])
print(f'Number of parametrs: {num_parsms}')

trainable_params = sum([ p.numel() for p in model_pt.parameters() if p.requires_grad ])
print(f'Number of trainable parametrs: {trainable_params}')

non_trainable_params = sum([ p.numel() for p in model_pt.parameters() if not p.requires_grad ])
print(f'Number of non trainable parametrs: {non_trainable_params}')

Number of parametrs: 34307
Number of trainable parametrs: 34307
Number of non trainable parametrs: 0


In [52]:
# Define the loss function
loss_fn = nn.CrossEntropyLoss()

# Define the optimizer
optimizer = torch.optim.Adam(
    params=model_pt.parameters(),
    lr=best_config['learning_rate']
)

In [53]:
seed = TRAINING_CONFIG['random_seed']
np.random.seed(seed=seed)
torch.manual_seed(seed=seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

g = torch.Generator()
g.manual_seed(42)

# Setup a train loader
train_loader = DataLoader(
    train_dataset_pt,
    batch_size=TRAINING_CONFIG['batch_size'],
    shuffle=True,
    generator=g
)

In [56]:
# Train the model
for epoch in range(TRAINING_CONFIG['max_epochs']):
    model_pt.train().to(device)

    for X, y in train_loader:
        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits = model_pt(X)
        loss = loss_fn(logits, y)

        loss.backward()
        optimizer.step()

    print(f'Epoch: {epoch + 1} | loss: {loss.item()}')

Epoch: 1 | loss: 0.5559282898902893
Epoch: 2 | loss: 0.5180783867835999
Epoch: 3 | loss: 0.4351341128349304


In [57]:
# setup the test dataloader
test_loader = DataLoader(
    test_dataset_pt,
    batch_size=TRAINING_CONFIG['batch_size'],
    shuffle=False
)

# Put the model in eval mode
model_pt.eval()

total_test_loss = 0.
correct = 0
test_examples = 0

with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)
        y = y.to(device)

        logits = model_pt(X)
        loss = loss_fn(logits, y)

        batch_size = y.size(0)

        total_test_loss += loss.item() * batch_size
        correct += (
            torch.argmax(logits, dim=1) == y
        ).sum().item()

        test_examples += batch_size

final_test_loss = total_test_loss / test_examples
final_test_accuracy = correct / test_examples


print(f'Final test loss: {final_test_loss}')
print(f'Final test accuracy: {final_test_accuracy}')

Final test loss: 0.4046264390150706
Final test accuracy: 0.848


In [58]:
# Let's save our final model
torch.save(
    {
        'model_state_dict' : model_pt.state_dict(),
        'config' : best_config,
        'input_dim' : INPUT_DIM,
        'num_classes' : NUM_CLASSES
    },
    '/tmp/final_torch_model.pt'
)